# 3-Amaliy Mashg'ulot: Ma’lumotlar to‘plami, atributlar va ma’lumotlar turlarini aniqlash hamda tahlil qilish
**Fan:** Data Mining  
**Mavzu:** «Talabalar» ma'lumotlar to'plami asosida atribut turlarini ajratish, statistik ko'rsatkichlarni hisoblash va ma'lumotlar sifatini (outlier, missing values) tekshirish.

In [ ]:
# 1. Kutubxonalarni import qilish va datasetni yuklash
import os
import pandas as pd
import numpy as np

df = pd.read_csv('talabalar.csv')
print('Dataset hajmi:', df.shape)
display(df)

## 1-topshiriq. Ma'lumotlar to'plamini tavsiflash
1. **Obyektlar va atributlar:** 10 ta obyekt (satr) va 10 ta atribut (ustun) mavjud.
2. **Obyekt:** Har bir satr alohida bir talabani ifodalaydi.
3. **Identifikator:** `ID` atributi faqat tartib raqami bo'lib, tahlilda qatnashmaydi.
4. **Amaliy savollar:**
   - Stipendiya olish GPA ko'rsatkichiga bog'liqmi?
   - Qaysi ta'lim yo'nalishi talabalarida eng yuqori GPA kuzatiladi?

## 2-topshiriq. Atribut turlarini aniqlash jadvali
| Atribut | Shkalasi | Turi | Asoslash |
| :--- | :--- | :--- | :--- |
| `Jinsi` | Binar (simmetrik) | Diskret | Faqat 2 ta toifa: Erkak / Ayol |
| `Yoshi` | Nisbat (Ratio) | Diskret | Mutlaq nol bor, 20 yosh 10 yoshdan 2 marta katta |
| `Kurs` | Ordinal | Diskret | 1 < 2 < 3 < 4 tartib va ierarxiya mavjud |
| `Yo‘nalish` | Nominal | Diskret | Matematik tartiblanmaydigan soha nomlari |
| `GPA` | Nisbat (Ratio) | Uzluksiz | Mutlaq nol bor (0.0), kasr sonli qiymatlar |
| `Stipendiya` | Binar (asimmetrik) | Diskret | Faqat 2 ta holat: Ha / Yo'q |
| `Baho darajasi`| Ordinal | Diskret | Qoniqarli < Yaxshi < A'lo darajasi |
| `Tug‘ilgan yil`| Interval | Diskret | Taqvimiy shkala, mutlaq nol mavjud emas |

## 3-topshiriq. Atributlarning statistik tavsifi

In [ ]:
# Sonli atributlar statistikasi (Yoshi va GPA)
yosh_all = df['Yoshi'].dropna()
yosh_clean = df[df['Yoshi'] < 100]['Yoshi'].dropna()
gpa = df['GPA'].dropna()

stats_df = pd.DataFrame({
    'Yoshi (120 bilan)': [yosh_all.min(), yosh_all.max(), yosh_all.max() - yosh_all.min(), round(yosh_all.mean(), 2), yosh_all.median(), yosh_all.mode()[0]],
    'Yoshi (Tozalangan)': [yosh_clean.min(), yosh_clean.max(), yosh_clean.max() - yosh_clean.min(), round(yosh_clean.mean(), 2), yosh_clean.median(), yosh_clean.mode()[0]],
    'GPA (9 ta talaba)': [gpa.min(), gpa.max(), round(gpa.max() - gpa.min(), 2), round(gpa.mean(), 2), gpa.median(), 'Moda yo\'q']
}, index=['Min', 'Max', 'Diapazon', 'O\'rtacha', 'Mediana', 'Moda'])

display(stats_df)

In [ ]:
# Toifaviy atributlar: Yo'nalish, Stipendiya, Baho darajasi
print('Yo\'nalish chastotasi va modasi:')
display(df['Yo‘nalish'].value_counts())
print('Moda:', df['Yo‘nalish'].mode()[0])

print('\nStipendiya chastotasi:')
display(df['Stipendiya'].value_counts())

print('\nBaho darajasi chastotasi va modasi:')
display(df['Baho darajasi'].value_counts())
print('Moda:', df['Baho darajasi'].mode()[0])
print('Mediana: Yaxshi (chunki o\'rtadagi 5 va 6-o\'rindagi qiymat)')

## 4-topshiriq. Python (pandas) orqali tahlil va sifatni tekshirish

In [ ]:
# Pandas tekshiruvlari
print('1. Hajmi:', df.shape)
print('\n2. Ma\'lumot turlari:')
display(df.dtypes)
print('\n3. Sonli ko\'rsatkichlar:')
display(df.describe())
print('\n4. Bo\'sh qiymatlar soni:')
display(df.isnull().sum())

In [ ]:
# Bo'sh qiymatlarni to'ldirish va Outlierni to'g'irlash
df_clean = df.copy()

# 1. Bekzodning GPA sini 'Yaxshi' baholilar o'rtachasi bilan to'ldirish
mean_yaxshi_gpa = df_clean[df_clean['Baho darajasi'] == 'Yaxshi']['GPA'].mean()
df_clean['GPA'] = df_clean['GPA'].fillna(round(mean_yaxshi_gpa, 2))

# 2. Madinaning Yo'nalishini moda bilan to'ldirish
df_clean['Yo‘nalish'] = df_clean['Yo‘nalish'].fillna(df_clean['Yo‘nalish'].mode()[0])

# 3. Otabekning yoshidagi 120 anomaliyasini 20 yoshga to'g'irlash (2005-yil tug'ilgan)
df_clean.loc[df_clean['ID'] == 9, 'Yoshi'] = 20

print('Tozalangan jadval:')
display(df_clean)